In [1]:
import os, sys, shutil, subprocess
from multiprocessing import cpu_count

print("Python:", sys.version)
print("CPU cores:", cpu_count())

# Kaggle paden
BASE = "/kaggle/working"
APPLIO_DIR = f"{BASE}/Applio"
LOGS_DIR = f"{APPLIO_DIR}/logs"

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
CPU cores: 4


In [2]:
APPLIO_REF = "3.6.2"   # probeer bv "main" of een release-tag als dat beter werkt

%cd /kaggle/working
if not os.path.isdir(APPLIO_DIR):
    !git clone https://github.com/IAHispano/Applio --branch {APPLIO_REF} --single-branch

# System deps (portaudio is vaak nodig)
!apt-get update -y
!apt-get install -y portaudio19-dev psmisc

%cd {APPLIO_DIR}

# Snelle/strakke pip via uv (zoals Applio notebooks doen)
!pip -q install uv
!uv pip install -q -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match --system

# Download Applio prerequisites (models/pretraineds etc.)
!python core.py prerequisites --models "True" --pretraineds_hifigan "True" > /dev/null 2>&1

print("✅ Install klaar")

/kaggle/working
Cloning into 'Applio'...
remote: Enumerating objects: 19967, done.
remote: Counting objects: 100% (144/144), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 19967 (delta 101), reused 83 (delta 71), pack-reused 19823 (from 2)
Receiving objects: 100% (19967/19967), 49.15 MiB | 37.70 MiB/s, done.
Resolving deltas: 100% (13073/13073), done.
Note: switching to 'c7be7282af0f7b6382cbb2ac780b11242da12916'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

Hit:1 http://archive.ubuntu.com/ubuntu jammy

In [3]:
!python /kaggle/working/Applio/core.py preprocess \
  --model_name "hitsugi_rvc_v2" \
  --dataset_path "/kaggle/input/datasets/michaeltje26/hitsugi" \
  --sample_rate "40000" \
  --cpu_cores "4" \
  --cut_preprocess "Automatic" \
  --process_effects "False" \
  --noise_reduction "False" \
  --chunk_len "3" \
  --overlap_len "0.3" \
  --normalization_mode "none"

Starting preprocess with 4 processes...
100%|█████████████████████████████████████████████| 4/4 [00:31<00:00,  7.77s/it]
Preprocess completed in 31.10 seconds on 00:34:24 seconds of audio.


In [4]:
!python /kaggle/working/Applio/core.py extract \
  --model_name "hitsugi_rvc_v2" \
  --f0_method "rmvpe" \
  --sample_rate "40000" \
  --cpu_cores "4" \
  --gpu "0" \
  --embedder_model "contentvec" \
  --embedder_model_custom "" \
  --include_mutes "2"

Starting pitch extraction on cuda:0 using rmvpe...
100%|█████████████████████████████████████████| 645/645 [00:16<00:00, 38.45it/s]
Pitch extraction completed in 24.41 seconds.
Starting embedding extraction with 4 cores on cuda:0...
100%|█████████████████████████████████████████| 645/645 [00:10<00:00, 61.28it/s]
Embedding extraction completed in 17.22 seconds.


In [5]:
!python /kaggle/working/Applio/core.py index \
  --model_name "hitsugi_rvc_v2" \
  --index_algorithm "Auto"

Saved index file '/kaggle/working/Applio/logs/hitsugi_rvc_v2/hitsugi_rvc_v2.index'


In [6]:
!python /kaggle/working/Applio/core.py train \
  --model_name "hitsugi_rvc_v2" \
  --save_every_epoch "20" \
  --save_only_latest "True" \
  --save_every_weights "False" \
  --total_epoch "200" \
  --sample_rate "40000" \
  --batch_size "10" \
  --gpu 0 \
  --pretrained "True" \
  --custom_pretrained "False" \
  --overtraining_detector "False" \
  --cleanup "False" \
  --cache_data_in_gpu "False" \
  --vocoder "HiFi-GAN" \
  --checkpointing "False"

2026-03-07 20:17:41.810643: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772914661.993874    1174 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772914662.049745    1174 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772914662.472689    1174 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772914662.472732    1174 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772914662.472736    1174 computation_placer.cc:177] computation placer alr

In [7]:
import os

paths = [
    "/kaggle/working/hitsugi_rvc_v2_inference.zip",
    "/kaggle/working/Applio/logs/hitsugi_rvc_v2/wavs",
    "/kaggle/working/Applio/logs/hitsugi_rvc_v2/raw",
    "/kaggle/working/Applio/logs/hitsugi_rvc_v2/f0",
    "/kaggle/working/Applio/logs/hitsugi_rvc_v2/features",
    "/kaggle/working/Applio/logs/hitsugi_rvc_v2/tensorboard"
]

for p in paths:
    if os.path.exists(p):
        if os.path.isdir(p):
            import shutil
            shutil.rmtree(p)
        else:
            os.remove(p)
        print("Deleted:", p)


Deleted: /kaggle/working/Applio/logs/hitsugi_rvc_v2/f0


In [10]:
# dit is een pth export met 18 log configs net als die MM pth

import torch, json
from pathlib import Path
from collections import OrderedDict

RUN_DIR = Path("/kaggle/working/Applio/logs/hitsugi_rvc_v2")

g_list = sorted(RUN_DIR.glob("G_*.pth"), key=lambda p: p.stat().st_mtime, reverse=True)
if not g_list:
    raise FileNotFoundError(f"Geen G_*.pth gevonden in {RUN_DIR}")
G_PATH = g_list[0]

ckpt = torch.load(G_PATH, map_location="cpu")

if isinstance(ckpt, dict) and "model" in ckpt:
    weight = ckpt["model"]
elif isinstance(ckpt, dict) and "weight" in ckpt:
    weight = ckpt["weight"]
else:
    weight = ckpt

f0_val = 1
cfg_path = RUN_DIR / "config.json"
if cfg_path.exists():
    cfg_json = json.loads(cfg_path.read_text())
    f0_val = int(cfg_json.get("f0", 1))

config18 = [
    1025,
    32,
    192,
    192,
    768,
    2,
    6,
    3,
    0,
    "1",
    [3, 7, 11],
    [[1, 3, 5], [1, 3, 5], [1, 3, 5]],
    [10, 10, 2, 2],   # <- belangrijk
    512,
    [16, 16, 4, 4],   # <- belangrijk
    109,
    256,
    40000
]

export = OrderedDict({
    "weight": weight,
    "config": config18,
    "info": "hitsugi_fixed_exact",
    "sr": "40k",
    "f0": f0_val,
    "version": "v2",
})

OUT = RUN_DIR / "hitsugi_fixed_exact_40k.pth"
torch.save(export, OUT)

print("Exported:", OUT)
print("config len:", len(export["config"]))
print("config[12] upsample_rates:", export["config"][12])
print("config[14] upsample_kernel_sizes:", export["config"][14])
print("config[-1] sr:", export["config"][-1])
print("version:", export["version"])

Exported: /kaggle/working/Applio/logs/hitsugi_rvc_v2/hitsugi_fixed_exact_40k.pth
config len: 18
config[12] upsample_rates: [10, 10, 2, 2]
config[14] upsample_kernel_sizes: [16, 16, 4, 4]
config[-1] sr: 40000
version: v2


In [ ]:
# dit is de combined index maker voor een pth download op de vdl server

%%bash
cd /kaggle/working/Applio/logs/hitsugi_rvc_v2

# combineer pth achter index
cat hitsugi_rvc_v2.index hitsugi_rvc_v2_export.pth > combined.index

ls -lh combined.index